# Quadratic Hawkes (exp synapse): pipeline theory vs simulation

Slim version of `hawkes_td_only_quad_expg.ipynb` where the entire
theory side has been replaced by **one call** to
`pipeline.compute_cumulants(...)` operating on the standardized
theory file `pipeline.theories.quad_hawkes_2pop_expg`.  The
simulation side is unchanged: it uses
`models.hawkes_sim_quad_expg_numba` directly and the factorial
cumulant estimator from `models.cumulant_estimator`.

Why two notebooks?  The old quad_expg notebook is the explicit
walkthrough — useful for teaching the framework, expensive to
edit and easy to break.  This one trades visibility for speed:
swap the `theory` import to a different `pipeline.theories.*`
module and you get the corresponding sim/theory comparison
for free.


## 1. Setup

In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, time
import numpy as np
import matplotlib.pyplot as plt

# Pipeline (theory side) — does NOT contain any simulation code
sys.path.insert(0, os.path.abspath('..'))
from pipeline import compute_cumulants
from pipeline.theories.quad_hawkes_2pop_expg import HAWKES_MODEL

# Sim side — model-specific, lives in models/
from models.hawkes_sim_quad_expg_numba import sim_hawkes_quad_expg_numba
from models.cumulant_estimator import compute_kpoint_slice


## 2. Configuration

Theory and simulation share a single `fundamental` dict.  The
simulator picks the parameters it needs by name (`E`, `w`, `tau`,
`a`, `tau_g`); the pipeline does the same for the theory side.


In [ ]:
fundamental = {
    'E':     [0.5, 0.5],
    'w':     [[0.3, 0.25], [0.3, 0.35]],
    'tau':   10.0,
    'a':     0.44,
    'tau_g': 5.0,
}

# k-point cumulant + loop order
k        = 2
max_ell  = 0
external_fields = [('n', 1), ('n', 2)]   # 'n' is the natural name; pipeline maps 'n' → internal 'dn'

# τ grid (theory + sim share this)
tau_max  = 50.0
tau_step = 0.5

# Pipeline parallelism — fanned across enumeration (step [5]) and
# Phase J τ-grid evaluation (step [7]).  Set N_WORKERS=None to let
# the pipeline pick min(os.cpu_count(), n_tasks); set it explicitly
# to cap process count on shared / loaded machines.  PARALLEL=False
# forces serial (useful for debugging).
PARALLEL  = True
N_WORKERS = None

# Simulation
N_RUNS   = 5
T_sim    = float(5_000_000)    # total time per run (rate × T sets the
                                # signal-to-shot-noise ratio of the
                                # estimator)
dt_sim   = 0.01                 # Euler step
dt_bin   = 0.25                 # binning resolution for the cumulant
                                # estimator (smaller → less bin bias,
                                # larger → fewer bins to FFT through)

print(f'k={k}, max_ell={max_ell}, external_fields={external_fields}')
print(f'tau_max={tau_max}, tau_step={tau_step}')
print(f'PARALLEL={PARALLEL}, N_WORKERS={N_WORKERS}')
print(f'N_RUNS={N_RUNS}, T_sim={T_sim:.0g}, dt_sim={dt_sim}, dt_bin={dt_bin}')


## 3. Theory side — one pipeline call

`compute_cumulants` runs the entire theoretical pipeline:
FieldTheory.expand → propagator construction → MF self-consistency
→ diagram enumeration / typing / dedup → Phase J integration on
the τ grid.  It returns a `result` dict with `total_C` callable,
the τ-grid evaluation, MF values, and the per-diagram records.


In [ ]:
# Single compute_cumulants call — returns per-loop-order curves in
# th['C_tau_by_ell'].  C_tau itself is the sum across all ells.

t0 = time.perf_counter()
th = compute_cumulants(
    model           = HAWKES_MODEL,
    k               = k,
    max_ell         = max_ell,
    fundamental     = fundamental,
    external_fields = external_fields,
    tau_max         = tau_max,
    tau_step        = tau_step,
    parallel        = PARALLEL,
    n_workers       = N_WORKERS,
    verbose         = True,
)
print(f'\nTheory side took {time.perf_counter() - t0:.1f}s')

tau_grid_th    = th['tau_grid']
C_theory_total = th['C_tau'].real
C_by_ell       = th['C_tau_by_ell']     # {0: tree, 1: 1-loop, ...}
C_theory_tree  = C_by_ell[0].real if 0 in C_by_ell else np.zeros_like(C_theory_total)
C_theory_loop  = C_theory_total - C_theory_tree

# Mean-field accessor — natural physical names ('n', 'v', 'm') with
# 1-based population indexing.  ``mf['n', 1]`` returns n*_1, etc.
mf    = th['mf']
nstar = mf['n']               # whole vector
vstar = mf['v']
print(f'\nMF: ' + ', '.join(f'{k_}={mf[k_]}' for k_ in mf))
for i in range(len(nstar)):
    print(f'   pop {i+1}:  n* = {mf["n", i+1]:.6f},  '
          f'v* = {mf["v", i+1]:.6f}')
print(f'  Total diagrams: {len(th["diagrams"])}')
n_per_ell = {ell: sum(1 for r in th['diagrams'] if r['ell'] == ell)
             for ell in sorted({r['ell'] for r in th['diagrams']})}
for ell, n_d in n_per_ell.items():
    print(f'    ell={ell}: {n_d} diagrams')


## 4. Simulation side

Runs the existing numba simulator `N_RUNS` independent times with
fresh seeds, bins each run's spike counts, and averages the
factorial cumulant slice across runs.  This is exactly the path the
long-form quad_expg notebook uses — only the boilerplate around it
is leaner here.


In [ ]:
import secrets as _secrets

E_sim     = [float(x) for x in fundamental['E']]
w_sim     = [[float(x) for x in row] for row in fundamental['w']]
tau_sim   = float(fundamental['tau'])
tau_g_sim = float(fundamental['tau_g'])
a_sim     = float(fundamental['a'])
npop_sim  = len(E_sim)

n_steps        = int(T_sim / dt_sim)
bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
dt_bin_eff     = bin_size_steps * dt_sim
n_bins         = n_steps // bin_size_steps
max_lag_bins   = int(tau_max / dt_bin_eff)
tau_sim_grid   = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin_eff

v_init = np.array(vstar, dtype=float)
E_arr  = np.array(E_sim,  dtype=float)
W_arr  = np.array(w_sim,  dtype=float)

# JIT warmup
_ = sim_hawkes_quad_expg_numba(
    int(1000), float(dt_sim), float(tau_sim),
    float(tau_g_sim), float(a_sim),
    E_arr, W_arr, v_init.copy(),
    int(bin_size_steps), int(100), int(0),
)

BASE_SEED   = _secrets.randbits(31)
pop_indices = [ef[1] - 1 for ef in external_fields]
field_types = [ef[0] for ef in external_fields]

C_sim_runs = []
rate_runs  = []
t0 = time.perf_counter()
for run in range(N_RUNS):
    seed = int(BASE_SEED + run)
    binned_counts, voltage_bins, total_spikes = sim_hawkes_quad_expg_numba(
        int(n_steps), float(dt_sim), float(tau_sim),
        float(tau_g_sim), float(a_sim),
        E_arr, W_arr, v_init.copy(),
        int(bin_size_steps), int(n_bins), seed,
    )
    rate_runs.append(
        [float(total_spikes[i]) / T_sim for i in range(npop_sim)]
    )
    # k=2 single sweep slice: leg 0 fixed at 0, leg 1 sweeps
    tau_run, C_run = compute_kpoint_slice(
        binned_counts, float(dt_bin_eff),
        [int(p) for p in pop_indices],
        [0, None], int(max_lag_bins),
        field_types=field_types,
        voltage_bins=voltage_bins,
    )
    C_sim_runs.append(C_run)
    print(f'  run {run+1}/{N_RUNS}: rates = {rate_runs[-1]}')

C_sim_runs = np.array(C_sim_runs)
C_sim_mean = C_sim_runs.mean(axis=0)
C_sim_sem  = C_sim_runs.std(axis=0, ddof=1) / np.sqrt(N_RUNS)
rate_runs_mean = np.array(rate_runs).mean(axis=0)
print(f'\nSimulation side took {time.perf_counter() - t0:.1f}s '
      f'({N_RUNS} runs × T={T_sim:.0g})')
print(f'  Sim mean rates: {rate_runs_mean}')
print(f'  Theory n*:      {nstar}')


## 5. Theory vs simulation comparison

Top: side-by-side mean firing rates (sim vs MF).  Bottom: the
smooth $C^{(2)}(\tau)$ slice from theory overlaid on the
sim's averaged factorial cumulant slice.  Sim shading shows the
across-run SEM band — useful for telling sim noise from a
theory/sim mismatch.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         gridspec_kw={'width_ratios': [1, 2]})

# Mean firing rates (sim vs MF) — quick sanity bar plot
ax_bar = axes[0]
x = np.arange(npop_sim)
width = 0.35
ax_bar.bar(x - width/2, rate_runs_mean, width, label='Simulation',
           color='#2ECC71', edgecolor='black')
ax_bar.bar(x + width/2, nstar, width, label='Mean-field',
           color='#3498DB', edgecolor='black')
ax_bar.set_xticks(x)
ax_bar.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
ax_bar.set_ylabel('Firing rate')
ax_bar.set_title('Mean firing rates')
ax_bar.legend()
ax_bar.grid(True, axis='y', alpha=0.25)

# C^(2)(tau) slice
ax = axes[1]
ax.plot(tau_sim_grid, C_sim_mean, color='#1f1f1f', linewidth=1.4,
        label=f'Simulation ({N_RUNS} runs avg)', alpha=0.85)
ax.fill_between(tau_sim_grid,
                C_sim_mean - C_sim_sem,
                C_sim_mean + C_sim_sem,
                color='#1f1f1f', alpha=0.15)
# Tree (always plotted)
ax.plot(tau_grid_th, C_theory_tree, color='#3F00FF', linewidth=1.6,
        linestyle='--', label='Theory: tree only')
# Tree + loop (only when max_ell > 0; otherwise duplicate of tree)
if max_ell > 0:
    ax.plot(tau_grid_th, C_theory_total, color='#E74C3C', linewidth=1.6,
            label=f'Theory: tree + {max_ell}-loop')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
field_a, field_b = external_fields
ax.set_xlabel(r'$\tau_1$')
ax.set_ylabel(r'$C^{(2)}$')
ax.set_title(f'Cross-covariance: {field_a[0]}_{field_a[1]}, '
             f'{field_b[0]}_{field_b[1]}')
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


## 6. Numerical residual

Quantitative check: interpolate theory onto the sim's τ-grid and
report the per-τ residual.  RMS / max-abs of the residual relative
to the sim's peak gives a one-number health metric for the
comparison.


In [ ]:
# Residual is sim minus the FULL theory (tree + loop).  When max_ell=0
# this equals the tree-only residual; when max_ell>0 it folds in the
# loop correction so residual≈sim_SEM means "theory at this order
# explains sim within MC noise."
C_total_on_sim_grid = np.interp(tau_sim_grid, tau_grid_th, C_theory_total)
residual            = C_sim_mean - C_total_on_sim_grid

peak        = max(abs(C_sim_mean.max()), abs(C_sim_mean.min()))
rms_rel     = float(np.sqrt(np.mean(residual**2)) / peak)
max_abs_rel = float(np.max(np.abs(residual)) / peak)

print(f'Sim peak |C|             : {peak:+.4e}')
print(f'Residual RMS / peak      : {rms_rel:.3%}')
print(f'Residual max / peak      : {max_abs_rel:.3%}')
print(f'Sim SEM at peak          : '
      f'{C_sim_sem[np.argmax(np.abs(C_sim_mean))]:+.3e}')
print('(if residual ≈ sim SEM, theory and sim agree within sim noise)')

if max_ell > 0:
    # How much of the gap between sim and tree got closed by the loop?
    C_tree_on_sim_grid  = np.interp(tau_sim_grid, tau_grid_th, C_theory_tree)
    tree_residual       = C_sim_mean - C_tree_on_sim_grid
    tree_rms_rel        = float(np.sqrt(np.mean(tree_residual**2)) / peak)
    print()
    print(f'Tree-only residual RMS / peak  : {tree_rms_rel:.3%}')
    print(f'Tree+loop residual RMS / peak  : {rms_rel:.3%}'
          f'   ← Δ = {tree_rms_rel - rms_rel:+.3%}')


## 7. (Optional) Save and PDF report

Drop the theory result + the sim slice to disk for later analysis,
and generate the multi-page PDF report from the pipeline.


In [ ]:
out_dir = '../pipeline_outputs/quad_expg_sim_compare'
os.makedirs(out_dir, exist_ok=True)

npz_path = f'{out_dir}/quad_expg_k{k}_ell{max_ell}.npz'
csv_path = f'{out_dir}/quad_expg_k{k}_ell{max_ell}.csv'

# Sim sidecar — extra keys merged into the same NPZ alongside the
# pipeline schema (C_tree, C_<n>_loop, C_total, mf_*, parameter_keys).
sim_extra = {
    'tau_grid_sim'    : tau_sim_grid,
    'C_sim_mean'      : C_sim_mean,
    'C_sim_sem'       : C_sim_sem,
    'rates_sim_mean'  : rate_runs_mean,
    'sim_N_RUNS'      : np.array([N_RUNS], dtype=int),
    'sim_T'           : np.array([T_sim]),
    'sim_dt'          : np.array([dt_sim]),
    'sim_dt_bin'      : np.array([dt_bin]),
}

from pipeline import save_npz, save_csv
save_npz(th, npz_path, extra=sim_extra)
save_csv(th, csv_path)    # CSV is theory-only (sim has its own grid)

print(f'Saved: {npz_path}')
print(f'Saved: {csv_path}')

# PDF diagram report
from pipeline import generate_report
pdf_path = f'{out_dir}/quad_expg_k{k}_ell{max_ell}_report.pdf'
generate_report(
    model           = HAWKES_MODEL,
    k               = k,
    fundamental     = fundamental,
    external_fields = external_fields,
    output_pdf      = pdf_path,
    result          = th,
    verbose         = False,
)
print(f'Saved: {pdf_path}')
